# Trusted Zone

## Initialization

In [5]:
import os
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, trim, when, udf
from pyspark.sql.types import StringType
import datetime
from bs4 import BeautifulSoup
from langdetect import detect

In [6]:
jdbc_jar_path = "../formatted_zone/postgresql-42.7.3.jar"
if 'spark' not in locals():
    if not os.path.exists(jdbc_jar_path):
         print(f"Error: JDBC Jar not found at {jdbc_jar_path}")
         # Handle error appropriately
    else:
        spark = SparkSession.builder \
            .appName("TrustedZoneETL") \
            .config("spark.jars", jdbc_jar_path) \
            .getOrCreate()
    print("Spark Session (re)started for Trusted Zone.")
else:
    print("Using existing Spark Session for Trusted Zone.")

Using existing Spark Session for Trusted Zone.


In [7]:
pg_url = "jdbc:postgresql://localhost:5432/bdaproject2"
pg_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

if isinstance(pg_properties["password"], bytes):
    pg_properties["password"] = pg_properties["password"].decode("utf-8", errors="replace")

## Movies Database (TMDB)

In [8]:
print("\nProcessing Movies Table (Trusted Zone)...")
try:
    formatted_movies_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movies") \
        .load()

    initial_count = formatted_movies_df.count()
    print(f"Read {initial_count} rows from formatted movies table.")

    # Data Quality que hem fet:
    # Completeness
    # Consistency: Budget, revenue, runtime, and vote counts cannot be negative.
    # Range Check: vote_average between 0 and 10.
    # Temporal Consistency: Release dates should not be too far in the future.
    # We assume that revenue = 0 and budget means missing values
    
    trusted_movies_df = formatted_movies_df.filter(
        col("movie_id").isNotNull() & \
        col("title").isNotNull() & (trim(col("title")) != "") & \
        col("release_date").isNotNull() & \
        (col("budget") > 0) & \
        (col("revenue") > 0) & \
        (col("runtime") >= 0) & \
        (col("vote_count") >= 0) & \
        (col("vote_average").between(0, 10)) & \
        (col("release_date") < datetime.date(2030, 1, 1))
    )

    trusted_movies_df.cache()
    final_count = trusted_movies_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to movies table.")

    trusted_movies_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movies") \
        .mode("overwrite") \
        .save()

    print("Movies table quality checked and written to PostgreSQL (Trusted).")
    trusted_movies_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing Movies table in Trusted Zone: {e}")


Processing Movies Table (Trusted Zone)...
Read 2877 rows from formatted movies table.
Applying DQ rules removed 1533 rows.
Writing 1344 trusted rows back to movies table.


25/06/07 16:50:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Movies table quality checked and written to PostgreSQL (Trusted).
+--------+--------------------+----------------------+--------------------------------------------------+------------+--------+---------+-------+----------+------------+----------+--------+--------------------------------------------------+-----+--------------------------------+---------+-----------------+-------------+-----------------------------+
|movie_id|               title|        original_title|                                          overview|release_date|  budget|  revenue|runtime|popularity|vote_average|vote_count|  status|                                           tagline|adult|                     poster_path|  imdb_id|original_language|collection_id|              collection_name|
+--------+--------------------+----------------------+--------------------------------------------------+------------+--------+---------+-------+----------+------------+----------+--------+-----------------------------------------

In [9]:
print("\nProcessing Movie Genres Table (Trusted Zone)...")
try:
    formatted_genres_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_genres") \
        .load()

    initial_count = formatted_genres_df.count()
    print(f"Read {initial_count} rows from formatted movie_genres table.")

    trusted_genres_df = formatted_genres_df.filter(
        col("movie_id").isNotNull() & \
        col("genre_id").isNotNull() & \
        col("genre_name").isNotNull() & (trim(col("genre_name")) != "")
    )

    trusted_genres_df.cache()
    final_count = trusted_genres_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to movie_genres table.")

    trusted_genres_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_genres") \
        .mode("overwrite") \
        .save()

    print("Movie Genres table quality checked and written to PostgreSQL (Trusted).")
    trusted_genres_df.show(5)

except Exception as e:
    print(f"Error processing Movie Genres table in Trusted Zone: {e}")


Processing Movie Genres Table (Trusted Zone)...
Read 6637 rows from formatted movie_genres table.
Applying DQ rules removed 0 rows.
Writing 6637 trusted rows back to movie_genres table.
Movie Genres table quality checked and written to PostgreSQL (Trusted).
+--------+--------+---------------+
|movie_id|genre_id|     genre_name|
+--------+--------+---------------+
|      78|     878|Science Fiction|
|      78|      18|          Drama|
|      78|      53|       Thriller|
|    2486|      14|        Fantasy|
|    2486|      28|         Action|
+--------+--------+---------------+
only showing top 5 rows



In [10]:
print("\nProcessing Movie Production Companies Table (Trusted Zone)...")
try:
    formatted_companies_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_production_companies") \
        .load()

    initial_count = formatted_companies_df.count()
    print(f"Read {initial_count} rows from formatted movie_production_companies table.")

    # Data Quality Rules (Referential Integrity & Completeness):
    trusted_companies_df = formatted_companies_df.filter(
        col("movie_id").isNotNull() & \
        col("company_id").isNotNull() & \
        col("company_name").isNotNull() & (trim(col("company_name")) != "")
    )

    trusted_companies_df.cache()
    final_count = trusted_companies_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to movie_production_companies table.")

    trusted_companies_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_production_companies") \
        .mode("overwrite") \
        .save()

    print("Movie Production Companies table quality checked and written to PostgreSQL (Trusted).")
    trusted_companies_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing Movie Production Companies table in Trusted Zone: {e}")


Processing Movie Production Companies Table (Trusted Zone)...
Read 7235 rows from formatted movie_production_companies table.
Applying DQ rules removed 0 rows.
Writing 7235 trusted rows back to movie_production_companies table.
Movie Production Companies table quality checked and written to PostgreSQL (Trusted).
+--------+----------+---------------------+--------------+
|movie_id|company_id|         company_name|origin_country|
+--------+----------+---------------------+--------------+
|      78|      5798|        Shaw Brothers|            HK|
|      78|      7965|     The Ladd Company|            US|
|      78|       174|Warner Bros. Pictures|            US|
|    2486|       711|    Fox 2000 Pictures|            US|
|    2486|      1302|  Davis Entertainment|            US|
+--------+----------+---------------------+--------------+
only showing top 5 rows



In [11]:
print("\nProcessing Movie Cast Table (Trusted Zone)...")
try:
    formatted_cast_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_cast") \
        .load()

    initial_count = formatted_cast_df.count()
    print(f"Read {initial_count} rows from formatted movie_cast table.")

    trusted_cast_df = formatted_cast_df.filter(
        col("credit_id").isNotNull() & (trim(col("credit_id")) != "") & \
        col("movie_id").isNotNull() & \
        col("actor_id").isNotNull() & \
        col("actor_name").isNotNull() & (trim(col("actor_name")) != "")
    )

    trusted_cast_df = trusted_cast_df.drop("profile_path")

    trusted_cast_df.cache()
    final_count = trusted_cast_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to movie_cast table.")

    trusted_cast_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_cast") \
        .mode("overwrite") \
        .save()

    print("Movie Cast table quality checked and written to PostgreSQL (Trusted).")
    trusted_cast_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing Movie Cast table in Trusted Zone: {e}")


Processing Movie Cast Table (Trusted Zone)...
Read 95548 rows from formatted movie_cast table.


Applying DQ rules removed 0 rows.
Writing 95548 trusted rows back to movie_cast table.


Movie Cast table quality checked and written to PostgreSQL (Trusted).
+------------------------+--------+--------+---------------+---------------+----------+------+
|               credit_id|movie_id|actor_id|     actor_name| character_name|cast_order|gender|
+------------------------+--------+--------+---------------+---------------+----------+------+
|52fe4240c3a36847f800fcd1|     411|    5528|William Moseley| Peter Pevensie|         0|     2|
|52fe4240c3a36847f800fcd5|     411|    5529|Anna Popplewell| Susan Pevensie|         1|     1|
|52fe4240c3a36847f800fccd|     411|    5527| Skandar Keynes|Edmund Pevensie|         2|     2|
|52fe4240c3a36847f800fcc9|     411|    5526| Georgie Henley|  Lucy Pevensie|         3|     1|
|52fe4240c3a36847f800fd09|     411|    3896|    Liam Neeson|  Aslan (voice)|         4|     2|
+------------------------+--------+--------+---------------+---------------+----------+------+
only showing top 5 rows



In [12]:
print("\nProcessing Movie Crew Table (Trusted Zone)...")
try:
    formatted_crew_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_crew") \
        .load()

    initial_count = formatted_crew_df.count()
    print(f"Read {initial_count} rows from formatted movie_crew table.")

    # Data Quality Rules (Primary Key, Foreign Key & Completeness):
    # credit_id, job, and department are essential crew information.
    trusted_crew_df = formatted_crew_df.filter(
        col("credit_id").isNotNull() & (trim(col("credit_id")) != "") & \
        col("movie_id").isNotNull() & \
        col("crew_member_id").isNotNull() & \
        col("crew_member_name").isNotNull() & (trim(col("crew_member_name")) != "") & \
        col("department").isNotNull() & (trim(col("department")) != "") & \
        col("crew_job").isNotNull() & (trim(col("crew_job")) != "")
    )

    trusted_crew_df = trusted_crew_df.drop("profile_path")

    trusted_crew_df.cache()
    final_count = trusted_crew_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to movie_crew table.")

    trusted_crew_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_crew") \
        .mode("overwrite") \
        .save()

    print("Movie Crew table quality checked and written to PostgreSQL (Trusted).")
    trusted_crew_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing Movie Crew table in Trusted Zone: {e}")


Processing Movie Crew Table (Trusted Zone)...
Read 152996 rows from formatted movie_crew table.


Applying DQ rules removed 0 rows.
Writing 152996 trusted rows back to movie_crew table.


Movie Crew table quality checked and written to PostgreSQL (Trusted).
+------------------------+--------+--------------+----------------+----------+------------------+------+
|               credit_id|movie_id|crew_member_id|crew_member_name|department|          crew_job|gender|
+------------------------+--------+--------------+----------------+----------+------------------+------+
|52fe4240c3a36847f800fcbf|     411|          5524|  Andrew Adamson| Directing|          Director|     2|
|52fe4240c3a36847f800fd2b|     411|          5541|   Philip Steuer|Production|          Producer|     2|
|52fe4240c3a36847f800fd31|     411|          5162|    Mark Johnson|Production|          Producer|     2|
|52fe4240c3a36847f800fd3d|     411|          5544|     Perry Moore|Production|Executive Producer|     2|
|52fe4240c3a36847f800fd43|     411|          5524|  Andrew Adamson|Production|Executive Producer|     2|
+------------------------+--------+--------------+----------------+----------+------------

## OMDB database
Movie reviews

In [38]:
print("\nProcessing OMDb Movie Details Table (Trusted Zone)...")
try:
    formatted_omdb_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "omdb_movie_details") \
        .load()

    initial_count = formatted_omdb_df.count()
    print(f"Read {initial_count} rows from formatted omdb_movie_details table.")

    trusted_omdb_df = formatted_omdb_df.filter(
        col("imdb_id").isNotNull() & (trim(col("imdb_id")) != "") & \
        col("title").isNotNull() & (trim(col("title")) != "") & \
        col("released").isNotNull() & \
        col("box_office").isNotNull() & (trim(col("box_office")) != "N/A") & \
        
        # Validity
        col("imdb_id").startswith("tt") & \
        (col("type") == 'movie') & \
        # There was only, one series -> it has different columns... To easy
        # our work, we prefer to assume we only have movies. 
        
        (col("year").isNotNull() & col("year").between(1888, 2030)) & \
        (col("metascore").isNull() | col("metascore").between(0, 100)) & \
        (col("imdb_rating").isNull() | col("imdb_rating").between(0.0, 10.0)) & \
        (col("imdb_votes").isNull() | (col("imdb_votes") >= 0)) & \

        (col("released") < datetime.date(2030, 1, 1))
    )

    trusted_omdb_df = trusted_omdb_df.drop("dvd_date", "production", "website")
    
    trusted_omdb_df.cache()
    final_count = trusted_omdb_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to omdb_movie_details table.")
    
    trusted_omdb_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "omdb_movie_details") \
        .mode("overwrite") \
        .save()

    print("OMDb Movie Details table quality checked and written to PostgreSQL (Trusted).")
    trusted_omdb_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing OMDb Movie Details table in Trusted Zone: {e}")
    import traceback
    traceback.print_exc()


Processing OMDb Movie Details Table (Trusted Zone)...


Read 1000 rows from formatted omdb_movie_details table.
Applying DQ rules removed 103 rows.
Writing 897 trusted rows back to omdb_movie_details table.
OMDb Movie Details table quality checked and written to PostgreSQL (Trusted).
+---------+-------------------------------------+----+-----+----------+-------+---------------------------+-----------------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+------------------------+-----------------------------------------+--------------------------------------------------+--------------------------------------------------+---------+-----------+----------+-----+------------+--------------------------------------------------+
|  imdb_id|                                title|year|rated|  released|runtime|                      genre|         director|                                            writer|                                            

## NYT Database

In [14]:
from pyspark.sql.functions import col, trim, to_date, current_date
import datetime

print("\nProcessing NYT Actor Articles Table (Trusted Zone)...")
try:
    raw_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "actor_articles") \
        .load()

    initial_count = raw_df.count()
    print(f"Loaded {initial_count} rows from actor_articles.")

    # Denial Constraints filtering
    trusted_df = raw_df.filter(
        # Completeness: Non-null, non-empty fields
        col("actor_id").isNotNull() & 
        (trim(col("title")) != "") &
        (trim(col("abstract")) != "") &
        (trim(col("web_url")) != "") &
        (trim(col("section")) != "") &
        col("pub_date").isNotNull() &

        # Consistency: pub_date should not be in the future
        (col("pub_date") <= current_date())
    )

    # Redundancy: actor_id and web_id should not be repeated (uniqueness)
    trusted_df = trusted_df.dropDuplicates(["actor_id", "web_url"])

    trusted_df.cache()
    final_count = trusted_df.count()
    print(f"Denial constraints removed {initial_count - final_count} rows.")

    # Overwrite the trusted version
    trusted_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "actor_articles") \
        .mode("overwrite") \
        .save()

    print(f"{final_count} trusted actor_articles written to PostgreSQL.")
    trusted_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing Trusted Zone: {e}")
    import traceback
    traceback.print_exc()



Processing NYT Actor Articles Table (Trusted Zone)...
Loaded 606 rows from actor_articles.


Denial constraints removed 0 rows.


606 trusted actor_articles written to PostgreSQL.
+--------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+----------+--------+
|actor_id|                                           web_url|                                             title|                                          abstract|  pub_date| section|
+--------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+----------+--------+
|     573|https://www.nytimes.com/2021/12/24/arts/music/b...|         Using a Pandemic Break to Tackle Bruckner|The Vienna Philharmonic records symphonies by A...|2021-12-24|    Arts|
|     418|https://www.nytimes.com/2023/06/08/movies/brook...|‘Brooklyn 45’ Review: A Little Something to Lif...|A group of World War II veterans unwisely perfo...|2023-06-08|  Movies|
|     955|https://www.nytimes.

## Spotify Database

In [ ]:
from pyspark.sql.functions import col, trim, to_date, current_date
import datetime

print("\nProcessing Spotify Music Tracks Table (Trusted Zone)...")
try:
    spotify_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "music_tracks") \
        .load()

    initial_count = spotify_df.count()
    print(f"Loaded {initial_count} rows from music_tracks.")

    # Apply Denial Constraints logic
    trusted_music_df = spotify_df.filter(
        # Completeness:
        col("track_id").isNotNull() &
        (trim(col("track_id")) != "") &
        (trim(col("track_name")) != "") &
        (trim(col("artist_name")) != "") &
        (trim(col("album_url")) != "") &

        # Consistency: 
        #   no negative nor null track durations,
        #   no negative nor null track numbers,
        #   no null popularity
        #   No null nor negative total tracks by album
        (col("track_duration_ms") > 0) &
        (col("track_number") > 0) &
        (col("track_popularity").isNotNull() ) &
        (col("album_total_tracks").isNull() | (col("album_total_tracks") > 0)) &


        # Completeness:
        col("album_release").isNotNull() &
        # Consistency: no album should be released in the future
        (col("album_release") <= current_date())
    )

    # Remove duplicate track_ids
    trusted_music_df = trusted_music_df.dropDuplicates(["track_id"])

    trusted_music_df.cache()
    final_count = trusted_music_df.count()
    print(f"Denial constraints removed {initial_count - final_count} rows.")

    # Write trusted version back
    trusted_music_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "music_tracks") \
        .mode("overwrite") \
        .save()

    print(f"{final_count} trusted music_tracks written to PostgreSQL.")
    trusted_music_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing Trusted Zone for music_tracks: {e}")
    import traceback
    traceback.print_exc()



Processing Spotify Music Tracks Table (Trusted Zone)...
Loaded 600 rows from music_tracks.
Denial constraints removed 600 rows.
0 trusted music_tracks written to PostgreSQL.
+--------+-----------+----------+-------------+---------+---------+------------------+----------+--------+-----------------+----------------+------------+-----------+---------+-----------------+------------+
|movie_id|movie_title|album_name|album_release|album_url|album_img|album_total_tracks|track_name|track_id|track_duration_ms|track_popularity|track_number|artist_name|artist_id|artist_popularity|album_genres|
+--------+-----------+----------+-------------+---------+---------+------------------+----------+--------+-----------------+----------------+------------+-----------+---------+-----------------+------------+
+--------+-----------+----------+-------------+---------+---------+------------------+----------+--------+-----------------+----------------+------------+-----------+---------+-----------------+-------